In [25]:
import os
import pandas as pd
from pathlib import Path
from PIL import Image

In [26]:
USER_ROOT = Path("/Users/romitbarua/BreastCancerResearch/Ekyaalo-WSI-ROI-Detection/")
DATA_ROOT = USER_ROOT / "data/raw/AnnotatedDataSet"
VALID_EXTENSIONS = ['.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp']

In [27]:

def generate_metadata(root_dir, label_map):

    # Get the classes from the folders
    classes = [d.name for d in DATA_ROOT.iterdir() if d.is_dir()]
    print('Classes:', classes)

    rows = []
    for cls in classes:
        class_dir = DATA_ROOT / cls

        assert cls in label_map, f"Class {cls} not found in label map"
        label = label_map[cls]

        for file in class_dir.iterdir():
            if not file.suffix in VALID_EXTENSIONS:
                print(f"Warning: invalid file extension: {file}")
                continue
            fpath = file.as_posix()
            fname = file.name
            extension = file.suffix

            slide_id = "UNKNOWN"
            if "Z-" in fname and ".tif" in fname:
                try:
                    slide_id = fname.split("Z-")[1].split(".tif")[0]
                except Exception:
                    slide_id = fname  # fallback
            else:
                slide_id = fname  # fallback if pattern different

            # Get image dimensions (height, width)
            try:
                with Image.open(fpath) as img:
                    width, height = img.size
            except Exception as e:
                print(f"Warning: Failed to get size for {fpath}: {e}")
                width, height = None, None

            # remove the USER_ROOT from the filepath
            fpath = str(fpath).replace(str(USER_ROOT), "")

            rows.append({
                "filepath": fpath,
                "fname": fname,
                "class_name": cls,
                "label": label,
                "slide_id": slide_id,
                "width": width,
                "height": height
            })

    df = pd.DataFrame(rows)
    print("Total images:", len(df))
    print(df["label"].value_counts().rename(index={0:"Benign",1:"Susp/Malig"}))
    
    return df


In [28]:
def assign_split(df, split_ratio=0.8):
    """
    Assigns splits to the dataframe based on the split ratio.
    """
    # Shuffle and split unique slide_ids to avoid leakage between splits
    slide_ids = df['slide_id'].unique()
    slide_ids = pd.Series(slide_ids).sample(frac=1, random_state=42)  # shuffle slide_ids

    n_unique = len(slide_ids)
    n_train = int(split_ratio * n_unique)

    train_slide_ids = set(slide_ids.iloc[:n_train])
    test_slide_ids = set(slide_ids.iloc[n_train:])

    df['split'] = df['slide_id'].apply(lambda x: 'train' if x in train_slide_ids else 'test')

    return df
    

In [29]:
df = generate_metadata(
    DATA_ROOT,
    {'Malignant': 1, 'Benign': 0, 'Suspicious': 1}
)
df = assign_split(df)
df.to_csv('/Users/romitbarua/BreastCancerResearch/Ekyaalo-WSI-ROI-Detection/data/metadata/metadata.csv')

Classes: ['Malignant', 'Benign', 'Suspicious']
Total images: 817
label
Susp/Malig    571
Benign        246
Name: count, dtype: int64
